In [ ]:
# Lab type: debug
# Course: AI403 — Building Production AI Agents
# Lesson: Handoffs, Delegation, and Where Context Is Lost
# Task: An orchestrator delegates drafting to a subagent. The handoff loses information in both
# directions. There are 3 bugs. After fixing each, write a one-sentence explanation below it.

# Lab: Lost in the Handoff

**How these labs work.** The "model" here is `scripted_model`, a plain Python function that makes the same choices an LLM would in this situation. That keeps every run deterministic and free, so each bug reproduces exactly. The harness around it — the loop, the tools, the checks — is real code, and it's what you'll be fixing.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
# No installs needed: this lab uses the Python standard library only.

In [ ]:
import json, re

USER_REQUEST = ("Draft apology emails for customers affected by last week's export outage. "
                "Do not draft anything for Enterprise customers: their account managers will "
                "contact them personally. Don't send anything yet - I'll review the drafts first.")

CUSTOMERS = {
    "C-1042": {"name": "Acme Corp", "plan": "Teams", "email": "ap@acme.example"},
    "C-2210": {"name": "Globex", "plan": "Enterprise", "email": "billing@globex.example"},
    "C-3307": {"name": "Initech", "plan": "Starter", "email": "finance@initech.example"},
    "C-4415": {"name": "Hooli", "plan": "Teams", "email": "accounts[at]hooli.example"},  # malformed
}
AFFECTED = ["C-1042", "C-2210", "C-3307", "C-4415"]
OUTBOX = []          # emails actually sent

def drafting_agent(instructions, inputs):
    """Stand-in for the drafting subagent. Like a capable model, it follows the instructions it
    was given - and only those. It reports what it skipped only if asked to."""
    text = instructions.lower()
    skip_enterprise = "enterprise" in text and ("do not" in text or "don't" in text)
    may_send = not ("none sent" in text or "do not send" in text or "don't send" in text)
    drafts, not_done = [], []
    for cid in inputs["customer_ids"]:
        c = CUSTOMERS[cid]
        if skip_enterprise and c["plan"] == "Enterprise":
            continue
        if not re.fullmatch(r"[^@\s]+@[^@\s]+", c["email"]):
            not_done.append(f"{cid}: email address malformed, no draft written")
            continue
        drafts.append(cid)
        if may_send:
            OUTBOX.append(cid)
    if "not_done" in text:
        return {"result": drafts, "not_done": not_done, "caveats": []}
    return f"Drafted apology emails for the affected customers ({len(drafts)} drafts)."

print(len(AFFECTED), "affected customers")

## The orchestrator under review

In [ ]:
# --- THE ORCHESTRATOR (review this code — is it correct?) ---
def orchestrator_note(request):
    """Stand-in for an LLM-written delegation note (kept short to save tokens)."""
    return "Write apology emails for each customer affected by the export outage."   # <- bug 1

def orchestrate(request):
    OUTBOX.clear()
    note = orchestrator_note(request)                                            # <- bug 2
    reply = drafting_agent(note, {"customer_ids": AFFECTED})
    return f"Done. {reply}"                                                        # <- bug 3

print(orchestrate(USER_REQUEST))
print("emails already sent to:", OUTBOX)

**Question 1.** Compare the orchestrator's reply with what actually happened. List every way the result differs from what the user asked for, and match each to one of the three `<-` markers.

In [ ]:
# Your answer (as comments):

<details>
<summary>🔑 Reveal answer — Question 1</summary>

- Globex (Enterprise) got a draft — and an email. The note paraphrased the goal and **dropped the Enterprise constraint** (bug 1: paraphrase loss).
- Emails were **sent**, although the user said not to send anything yet. The note never said what "done" means, so the subagent decided for itself (bug 2: no definition of done — and more generally, a note written by a model instead of a structured handoff).
- Hooli's malformed address meant no draft was written, but the reply says "drafted apology emails for the affected customers". The return trip carries a summary, not what was skipped (bug 3: the return path drops caveats), and the orchestrator repeats it as "Done".

</details>

## Fix the handoff

**Question 2.** Write a `Handoff` dataclass with `goal`, `original_request` (verbatim), `constraints` (list, copied verbatim), `inputs` (IDs), `done_when`, and `must_return` (default `["result", "not_done", "caveats"]`), plus an `instructions()` method that renders all of it as text for the subagent. Build one for this request.

In [ ]:
# Work here: Handoff

In [ ]:
# Bug 1 explanation:
# Bug 2 explanation:

<details>
<summary>🔑 Reveal answer — Question 2</summary>

```python
from dataclasses import dataclass, field

@dataclass
class Handoff:
    goal: str
    original_request: str
    constraints: list
    inputs: dict
    done_when: str
    must_return: list = field(default_factory=lambda: ["result", "not_done", "caveats"])

    def instructions(self):
        return "\n".join([
            f"Goal: {self.goal}",
            *[f"Constraint: {c}" for c in self.constraints],
            f"Done when: {self.done_when}",
            f"Return: {', '.join(self.must_return)}",
            f"Original request: {self.original_request}",
        ])

handoff = Handoff(
    goal="Draft apology emails for customers affected by the export outage.",
    original_request=USER_REQUEST,
    constraints=["Do not draft anything for Enterprise customers.", "Do not send anything."],
    inputs={"customer_ids": AFFECTED},
    done_when="One draft per eligible customer; none sent.")
print(handoff.instructions())
```

*Bug 1:* a paraphrase keeps the goal and loses the conditions, so constraints must be copied verbatim (and the original request passed along). *Bug 2:* without a definition of done — "none sent" — the receiver decides for itself what finishing means.

</details>

**Question 3.** Rewrite `orchestrate` to delegate with the `Handoff` and to report honestly: if the subagent returns anything in `not_done`, the orchestrator's reply must say so. Run it and check `OUTBOX`.

In [ ]:
# Work here: orchestrate with a structured handoff

In [ ]:
# Bug 3 explanation:

<details>
<summary>🔑 Reveal answer — Question 3</summary>

```python
def orchestrate(request):
    OUTBOX.clear()
    out = drafting_agent(handoff.instructions(), handoff.inputs)
    reply = f"Drafts written for {out['result']}."
    if out["not_done"]:
        reply += " Not done: " + "; ".join(out["not_done"])
    return reply

print(orchestrate(USER_REQUEST))
print("emails sent:", OUTBOX)
```

Drafts for Acme and Initech only, nothing sent, and the reply names Hooli's malformed address. *Bug 3:* the return trip is a handoff too — requiring `not_done` makes a partial result visible as partial.

</details>

**Question 4.** Write a constraint-propagation test: for each constraint the user stated, assert it appears verbatim in the handoff's instructions, and assert the *outcome* respects it (no Enterprise customer in the drafts; `OUTBOX` empty). Show that the original `orchestrator_note` fails the first check.

In [ ]:
# Work here: propagation test

<details>
<summary>🔑 Reveal answer — Question 4</summary>

```python
STATED = ["Do not draft anything for Enterprise customers", "Don't send anything yet"]

def propagation_problems(instructions, outcome):
    problems = [f"missing constraint: {c!r}" for c in STATED
                if c.lower().rstrip(".") not in instructions.lower()]
    enterprise = [cid for cid in outcome["result"] if CUSTOMERS[cid]["plan"] == "Enterprise"]
    if enterprise:
        problems.append(f"drafted for Enterprise customers {enterprise}")
    if OUTBOX:
        problems.append(f"sent emails {OUTBOX}")
    return problems or ["pass"]

OUTBOX.clear()
print("old note:", propagation_problems(orchestrator_note(USER_REQUEST), {"result": []}))
OUTBOX.clear()
outcome = drafting_agent(handoff.instructions(), handoff.inputs)
print("handoff: ", propagation_problems(handoff.instructions(), outcome))
```

The handoff carries the user's words in `original_request`, so both constraints are present verbatim even though the `constraints` field paraphrases one of them slightly — which is exactly why the original request travels with every handoff.

</details>

## Summary

1. A receiving agent knows only what its _______ contains.
2. Constraints are copied verbatim and the _______ request travels with every handoff.
3. The return trip must report what was _______, not just what was done.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **handoff**
2. **original**
3. **not done (skipped)**

</details>